# 10. Komunikace s databázovým systémem - Připojení, Ukládání a načítání dat, Mapování entit v OOP

### Připojení k RDBMS
* Připojení se realizuje přes knihovnu dodávanou přímo výrobcem databáze
* Přihlašovací údaje se do zdrojového kódu nepíší, načítají se dynamicky z externích konfiguračních souborů
* Vužívají se vytvářející návrhové vzory (např. Singleton, Lazy initialization)

### Operace C-R-U-D
* Veškerá manipulace nad tabulkou odpovídá základním operacím Create (příkaz `INSERT`), Read (`SELECT`), Update (`UPDATE`) a Delete (`DELETE`)
* Využívá se strukturální vzor Facade, hlavnímu programu nabídne pouze čisté a jednoduché rozhraní a SQL logiku vykoná skrytě
* Dalším přístupem je třívrstvá architektura (3-Tier)
* Při ukládání a načítání hrozí riziko SQL injection, vzniká nevalidovaným vstupem od uživatele
* Prepared statements a parametrizované dotazy tomu zabranují

### Mapování entit v OOP
* Objektově relační mapování (ORM) je k zrcadlení databázových řádků do objektových tříd
* Slouží k tomu aby vývojář nemusel psát manuálně SQL příkazy
* ORM může generovat více databázových dotazů než klasické ruční SQL
* Bez ORM lze mapování vyřešit návrhovými vzory
    * DAO (Data Access Object) / Table Gateway - pro jednu tabulku v DB existuje v programu přesně jedna třída, disponuje metodami jako `findByName()` nebo `getAll()` a veškerými CRUD
    * Active Record a Row Gateway - každý řádek z tabulky je v operační paměti reprezentován objektem,u sebe drží atributy a zároveň má metodu `save()`, kterou se zaktualizuje či vloží do DB

In [ ]:
import sqlite3
import json

**1. KONFIGURACE A PŘIPOJENÍ**

In [ ]:
# Načtení bezpečně z externího zdroje, nikoliv natvrdo v kódu
config_data = '{"db_file": "mojedb.sqlite"}'
nastaveni = json.loads(config_data)

class DatabazovePripojeni:
    _instance = None # Singleton - vždy drží jen jedno existující spojení

    @classmethod
    def get_instance(cls):
        # Lazy initialization - připojíme se k RDBMS až na vyžádání
        if cls._instance is None:
            # Použití přímé knihovny, obejití těžkého ORM frameworku
            cls._instance = sqlite3.connect(nastaveni["db_file"])
            kurzor = cls._instance.cursor()
            kurzor.execute('''CREATE TABLE IF NOT EXISTS UZIVATELE
                              (USERNAME TEXT, OBLIBENE_CISLO INT, OBLIBENA_BARVA TEXT)''')
            cls._instance.commit()
        return cls._instance

**2. MAPOVÁNÍ ENTIT A C-R-U-D OPERACE ZÁPISU**

In [ ]:
class UzivatelRadek:
    """Entita reprezentující jeden konkrétní záznam uživatele z tabulky"""
    def __init__(self, username, cislo, barva):
        self.username = username
        self.oblibene_cislo = cislo
        self.oblibena_barva = barva

    def save(self):
        # CREATE operace: Objekt má odpovědnost sám sebe bezpečně uložit
        db = DatabazovePripojeni.get_instance()
        kurzor = db.cursor()

        # Obrana proti SQL Injection pomocí parametrizovaného dotazu (znak ?)
        sql_dotaz = "INSERT INTO UZIVATELE VALUES (?, ?, ?)"
        kurzor.execute(sql_dotaz, (self.username, self.oblibene_cislo, self.oblibena_barva))
        db.commit()

**3. DATABÁZOVÁ VRSTVA A NAČÍTÁNÍ**

In [ ]:
class UzivateleDAO:
    """Třída starající se o hromadné dotazy (READ) nad celou tabulkou"""
    def nacti_vsechny_uzivatele(self):
        db = DatabazovePripojeni.get_instance()
        kurzor = db.cursor()
        kurzor.execute("SELECT * FROM UZIVATELE")
        surove_radky = kurzor.fetchall()

        # Namapování surových SQL řádků na žívé OOP objekty (Mapování entit)
        list_objektu = []
        for radek in surove_radky:
            list_objektu.append(UzivatelRadek(radek[0], radek[1], radek[2]))

        return list_objektu

**BĚH PROGRAMU**

In [ ]:
# Vložení záznamů (dle testovacího zadání v úloze 14.3)
novy_uzivatel1 = UzivatelRadek("Petra", 3, "modrá")
novy_uzivatel1.save() # Objekt se sám uloží

novy_uzivatel2 = UzivatelRadek("Vilém", 4, "růžová")
novy_uzivatel2.save()

# Načtení dat pomocí DAO (Read)
spravce_tabulky = UzivateleDAO()
vysledky = spravce_tabulky.nacti_vsechny_uzivatele()

print(f"Počet osobních účtů načtených přes DAO: {len(vysledky)}")
for entita in vysledky:
    print(f"- Uživatelská entita: Jméno: {entita.username}, Barva: {entita.oblibena_barva}")